In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q vllm
print("install done")

In [ ]:
# ============================================================================
# PHASE 10: EXTEND 16k TRACES TO 32k
# Resume truncated traces instead of regenerating -> saves ~4 GPU-hours
# ============================================================================
import json, os, re, time, glob, math
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset

os.makedirs("outputs", exist_ok=True)
MODEL = "WeiboAI/VibeThinker-3B"
BASE_LEN, TARGET = 16384, 32768
V1 = ("\n\n</think>\n\nI have reasoned enough. Based on the work above, "
      "the final answer is \\boxed{")

# ---- locate the traces file ------------------------------------------------
cands = (glob.glob("/kaggle/input/**/phase7_traces.json", recursive=True) +
         glob.glob("/kaggle/working/**/phase7_traces.json", recursive=True))
if not cands:
    raise FileNotFoundError(
        "phase7_traces.json not found. Did you add the Phase 7 notebook output "
        "as an Input? (right panel -> +Add Input -> Your Work -> Notebook Output)")
print("using traces:", cands[0])
d = json.load(open(cands[0]))
truths, traces = d["truths"], d["traces"]

ds = load_dataset("math-ai/aime25")["test"]
tok = AutoTokenizer.from_pretrained(MODEL)

llm = LLM(model=MODEL, dtype="float16", gpu_memory_utilization=0.90,
          max_model_len=34816, trust_remote_code=True)
print("\nModel loaded.\n")


def prompt_of(i):
    return tok.apply_chat_template(
        [{"role": "user", "content": ds[i]["problem"]}],
        tokenize=False, add_generation_prompt=True)


def boxed(t):
    i = t.rfind("\\boxed{")
    if i == -1: return None
    j, dep, o = i + 7, 1, []
    while j < len(t) and dep > 0:
        c = t[j]
        if c == "{": dep += 1
        elif c == "}":
            dep -= 1
            if dep == 0: break
        o.append(c); j += 1
    return "".join(o).strip()


def as_int(s):
    if s is None: return None
    m = re.search(r"-?\d+", s.replace(",", ""))
    return int(m.group()) if m else None


# ---- which traces need extending ------------------------------------------
todo = [(pi, ci) for pi in range(30) for ci in range(4)
        if traces[pi][ci]["finish"] != "stop"]
print(f"traces to extend: {len(todo)}  (the other {120-len(todo)} already finished)\n")

# full[pi][ci] = {"ids": [...], "text": str, "finish": str}
full = [[dict(traces[pi][ci]) for ci in range(4)] for pi in range(30)]

CONT = SamplingParams(n=1, temperature=1.0, top_p=0.95,
                      max_tokens=TARGET - BASE_LEN)

t_all = time.time()
CHUNK = 20
for s in range(0, len(todo), CHUNK):
    batch = todo[s:s + CHUNK]
    print(f"--- extending {s+1}-{s+len(batch)} of {len(todo)} ---")
    t0 = time.time()
    outs = llm.generate(
        [prompt_of(pi) + traces[pi][ci]["text"] for pi, ci in batch], CONT)
    dt = time.time() - t0
    newtok = sum(len(o.outputs[0].token_ids) for o in outs)
    fin = sum(1 for o in outs if o.outputs[0].finish_reason == "stop")
    print(f"    {dt/60:.1f} min, {newtok:,} new tokens, "
          f"{newtok/dt:.0f} tok/s, finished {fin}/{len(batch)}")

    for (pi, ci), o in zip(batch, outs):
        c = o.outputs[0]
        full[pi][ci] = {
            "ids": traces[pi][ci]["ids"] + list(c.token_ids),
            "text": traces[pi][ci]["text"] + c.text,
            "finish": c.finish_reason,
        }
    json.dump({"truths": truths, "traces": full},
              open("outputs/phase10_traces32k.json", "w"))
    print(f"    saved ({s+len(batch)}/{len(todo)})\n")

gen_min = (time.time() - t_all) / 60
print(f"extension complete: {gen_min:.1f} min\n")

# ---- fine-grained curve, now out to 32k -----------------------------------
print("=" * 74)
print("ACCURACY vs BUDGET (no forcing)")
print("=" * 74)
GRID = [16384, 20480, 24576, 28672, 32768]
curve = []
for B in GRID:
    ok = tr = 0
    for pi in range(30):
        for ci in range(4):
            t = full[pi][ci]
            if len(t["ids"]) <= B:
                txt, is_tr = t["text"], (t["finish"] != "stop")
            else:
                txt, is_tr = tok.decode(t["ids"][:B]), True
            tr += is_tr
            if as_int(boxed(txt)) == truths[pi]: ok += 1
    curve.append({"budget": B, "acc": round(100*ok/120, 1),
                  "trunc": round(100*tr/120, 1)})
    print(f"  {B:>6,} -> {100*ok/120:5.1f}%   truncated {100*tr/120:5.1f}%")
print(f"\n  Phase 5 independently measured 32k = 80.8%  <- validation check")
print()

# ---- forcing at 32k --------------------------------------------------------
jobs, base = [], []
for pi in range(30):
    row = []
    for ci in range(4):
        t = full[pi][ci]
        is_tr = t["finish"] != "stop"
        row.append(as_int(boxed(t["text"])))
        if is_tr:
            jobs.append((pi, ci, prompt_of(pi) + t["text"]))
    base.append(row)

print("=" * 74)
print(f"FORCING AT 32k  ({len(jobs)} truncated samples)")
print("=" * 74)
t0 = time.time()
fo = llm.generate([j[2] + V1 for j in jobs],
                  SamplingParams(n=1, temperature=0.0, max_tokens=24))
force_min = (time.time() - t0) / 60

b = c = 0
forced = [r[:] for r in base]
for (pi, ci, _), o in zip(jobs, fo):
    old, new = base[pi][ci], as_int(o.outputs[0].text)
    forced[pi][ci] = new
    o_ok, n_ok = (old == truths[pi]), (new == truths[pi])
    if not o_ok and n_ok: b += 1
    elif o_ok and not n_ok: c += 1

acc0 = sum(1 for pi in range(30) for p in base[pi] if p == truths[pi])
acc1 = sum(1 for pi in range(30) for p in forced[pi] if p == truths[pi])
n_disc = b + c
chi2 = ((abs(b-c)-1)**2)/n_disc if n_disc else 0.0
p = math.erfc(math.sqrt(chi2/2)) if chi2 > 0 else 1.0

print(f"  forcing took {force_min:.1f} min")
print(f"  no forcing : {100*acc0/120:5.1f}%")
print(f"  + forcing  : {100*acc1/120:5.1f}%   ({100*(acc1-acc0)/120:+.1f})")
print(f"  rescued b={b}   damaged c={c}   McNemar chi2={chi2:.2f}  p={p:.4f}")
print()

json.dump({"curve_16k_to_32k": curve,
           "forcing_32k": {"n_truncated": len(jobs), "acc_none": round(100*acc0/120,1),
                           "acc_forced": round(100*acc1/120,1),
                           "gain": round(100*(acc1-acc0)/120,1),
                           "rescued": b, "damaged": c,
                           "mcnemar_chi2": round(chi2,2), "p_value": round(p,4)},
           "gen_minutes": round(gen_min,1), "force_minutes": round(force_min,1)},
          open("outputs/phase10_results.json","w"), indent=2)

print("=" * 74)
print("COMPLETE RESULTS MATRIX")
print("=" * 74)
print(f"{'Budget':>8}{'No forcing':>13}{'+ Forcing':>12}{'Gain':>8}")
print("-" * 74)
for bud, nf, f_ in [("4k","30.0%","40.8%","+10.8"), ("8k","46.7%","55.8%","+9.1"),
                    ("16k","63.3%","70.8%","+7.5")]:
    print(f"{bud:>8}{nf:>13}{f_:>12}{'':>8}")
print(f"{'32k':>8}{100*acc0/120:>12.1f}%{100*acc1/120:>11.1f}%"
      f"{100*(acc1-acc0)/120:>+8.1f}")
print("-" * 74)
print(f"Total runtime: {gen_min+force_min:.0f} min")
print("Saved outputs/phase10_traces32k.json and phase10_results.json")

In [1]:
# ============================================================================
# DIAGNOSTIC: what is actually mounted?
# ============================================================================
import os, json

print("=" * 70)
print("EVERYTHING UNDER /kaggle/input")
print("=" * 70)
found = []
if not os.path.exists("/kaggle/input"):
    print("  /kaggle/input DOES NOT EXIST - no inputs attached at all")
else:
    n = 0
    for dirpath, dirnames, filenames in os.walk("/kaggle/input", followlinks=True):
        for fn in filenames:
            p = os.path.join(dirpath, fn)
            try:
                mb = os.path.getsize(p) / 1e6
            except OSError:
                mb = -1
            print(f"  {p}   ({mb:.1f} MB)")
            found.append(p)
            n += 1
            if n > 60:
                print("  ... (truncated)")
                break
        if n > 60:
            break
    if n == 0:
        print("  /kaggle/input exists but is EMPTY")

print()
print("=" * 70)
print("CANDIDATE TRACE FILES")
print("=" * 70)
cands = [p for p in found
         if p.endswith(".json") and ("trace" in p.lower() or "phase7" in p.lower())]
for p in cands:
    print(" ", p)
if not cands:
    print("  none found")

if cands:
    p = cands[0]
    print()
    print(f"Loading {p} ...")
    d = json.load(open(p))
    print("  top-level keys:", list(d.keys()))
    if "traces" in d:
        t = d["traces"]
        n_tr = sum(1 for a in range(len(t)) for b in range(len(t[a]))
                   if t[a][b]["finish"] != "stop")
        print(f"  problems={len(t)}  samples/problem={len(t[0])}")
        print(f"  truncated={n_tr}   (expect 58)")
        print("  >>> FILE IS GOOD. Use this path. <<<")

EVERYTHING UNDER /kaggle/input
  /kaggle/input/datasets/abuahmad3/vibethinker-phase7-dataset/phase7_traces (1).json   (11.7 MB)

CANDIDATE TRACE FILES
  /kaggle/input/datasets/abuahmad3/vibethinker-phase7-dataset/phase7_traces (1).json

Loading /kaggle/input/datasets/abuahmad3/vibethinker-phase7-dataset/phase7_traces (1).json ...
  top-level keys: ['truths', 'traces']
  problems=30  samples/problem=4
  truncated=58   (expect 58)
  >>> FILE IS GOOD. Use this path. <<<


In [1]:
# ============================================================================
# PHASE 11: UPPER BOUNDS   (CPU only, ~2 min, 0 GPU-hours)
# Closes the gate: a recovery rate is meaningless without a ceiling.
# ============================================================================
import os, re, json
from transformers import AutoTokenizer

os.makedirs("outputs", exist_ok=True)

# --- locate traces (os.walk, not glob - glob misses Kaggle symlink mounts) ---
hits = []
for root in ("/kaggle/input", "/kaggle/working"):
    if not os.path.exists(root):
        continue
    for dp, _, fns in os.walk(root, followlinks=True):
        for fn in fns:
            if fn.endswith(".json") and ("trace" in fn.lower() or "phase7" in fn.lower()):
                hits.append(os.path.join(dp, fn))
hits = sorted(set(hits), key=lambda p: -os.path.getsize(p))
assert hits, "traces not found"
print("using:", hits[0])

d = json.load(open(hits[0]))
truths, traces = d["truths"], d["traces"]
assert len(traces) == 30 and len(traces[0]) == 4, "unexpected shape"
tok = AutoTokenizer.from_pretrained("WeiboAI/VibeThinker-3B")

K = 4
FORCED = {4096: 40.8, 8192: 55.8, 16384: 70.8}   # measured, phases 6-7


def all_boxed(t):
    """Every \\boxed{...} in the trace, not just the last."""
    out, i = [], 0
    while True:
        i = t.find("\\boxed{", i)
        if i == -1:
            return out
        j, dep, buf = i + 7, 1, []
        while j < len(t) and dep > 0:
            ch = t[j]
            if ch == "{":
                dep += 1
            elif ch == "}":
                dep -= 1
                if dep == 0:
                    break
            buf.append(ch); j += 1
        out.append("".join(buf).strip()); i = j


def as_int(s):
    if s is None:
        return None
    m = re.search(r"-?\d+", s.replace(",", ""))
    return int(m.group()) if m else None


def prefix(t, B):
    if len(t["ids"]) <= B:
        return t["text"], (t["finish"] != "stop")
    return tok.decode(t["ids"][:B]), True


rows = []
print()
print("=" * 68)
print("UPPER BOUNDS")
print("=" * 68)
print(f"{'budget':>7}{'pass@1':>9}{'pass@4':>9}{'ceil_box':>10}"
      f"{'ceil_txt':>10}{'FP_ctrl':>9}{'forced':>9}")
print("-" * 68)

for B in sorted(FORCED):
    n_correct = n_any = 0
    n_tr = hit_box = hit_txt = hit_fp = 0
    for pi in range(30):
        truth = truths[pi]
        decoy = truths[(pi + 7) % 30]        # false-positive calibration
        solved_any = False
        for ci in range(K):
            txt, truncated = prefix(traces[pi][ci], B)
            boxes = all_boxed(txt)
            ok = (as_int(boxes[-1]) if boxes else None) == truth
            n_correct += ok
            solved_any |= ok
            if truncated:
                n_tr += 1
                hit_box += any(as_int(x) == truth for x in boxes)
                tail = txt[int(len(txt) * 0.8):]
                hit_txt += bool(re.search(rf"(?<!\d){truth}(?!\d)", tail))
                hit_fp  += bool(re.search(rf"(?<!\d){decoy}(?!\d)", tail))
        n_any += solved_any

    p1 = 100 * n_correct / (30 * K)
    p4 = 100 * n_any / 30
    cb = 100 * hit_box / n_tr if n_tr else 0.0
    ct = 100 * hit_txt / n_tr if n_tr else 0.0
    fp = 100 * hit_fp / n_tr if n_tr else 0.0
    print(f"{B//1024:>6}k{p1:>8.1f}%{p4:>8.1f}%{cb:>9.1f}%"
          f"{ct:>9.1f}%{fp:>8.1f}%{FORCED[B]:>8.1f}%")
    rows.append({"budget": B, "pass_at_1": round(p1,1), "pass_at_4": round(p4,1),
                 "ceiling_boxed_pct": round(cb,1), "ceiling_text_pct": round(ct,1),
                 "false_positive_ctrl_pct": round(fp,1),
                 "n_truncated": n_tr, "forced_pass_at_1": FORCED[B]})

print("-" * 68)
print("pass@4   oracle over the 4 samples (bounds any SELECTION method)")
print("ceil_box truncated traces where the TRUE answer appears in some \\boxed{}")
print("ceil_txt true answer appears as a standalone number in the last 20%")
print("FP_ctrl  same test with ANOTHER problem's answer -> false-positive rate")
print("         ceil_txt only means something insofar as it exceeds FP_ctrl")
print()

print("=" * 68)
print("HEADROOM - how much of the ceiling does forcing capture?")
print("=" * 68)
for r in rows:
    n_tr = r["n_truncated"]
    rescued = round((r["forced_pass_at_1"] - r["pass_at_1"]) / 100 * 120)
    ceiling = round(r["ceiling_boxed_pct"] / 100 * n_tr)
    frac = 100 * rescued / ceiling if ceiling else float("nan")
    print(f"  {r['budget']//1024:>3}k  truncated={n_tr:>3}  answer-in-trace={ceiling:>3}"
          f"  rescued={rescued:>3}  captured={frac:5.1f}%")
print()
print("captured > 70%  -> forcing is near-optimal; method is done")
print("captured < 40%  -> a better extractor has real headroom; future work")

json.dump({"upper_bounds": rows, "note": "CPU-derived; no GPU used"},
          open("outputs/phase11_upper_bounds.json", "w"), indent=2)
print("\nsaved outputs/phase11_upper_bounds.json")

using: /kaggle/input/datasets/abuahmad3/vibethinker-phase7-dataset/phase7_traces (1).json


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/616 [00:00<?, ?B/s]


UPPER BOUNDS
 budget   pass@1   pass@4  ceil_box  ceil_txt  FP_ctrl   forced
--------------------------------------------------------------------
     4k    30.0%    40.0%     20.0%     32.4%     2.9%    40.8%
     8k    46.7%    53.3%     21.0%     39.5%     4.9%    55.8%
    16k    63.3%    73.3%     24.1%     41.4%     3.4%    70.8%
--------------------------------------------------------------------
pass@4   oracle over the 4 samples (bounds any SELECTION method)
ceil_box truncated traces where the TRUE answer appears in some \boxed{}
ceil_txt true answer appears as a standalone number in the last 20%
FP_ctrl  same test with ANOTHER problem's answer -> false-positive rate
         ceil_txt only means something insofar as it exceeds FP_ctrl

HEADROOM - how much of the ceiling does forcing capture?
    4k  truncated=105  answer-in-trace= 21  rescued= 13  captured= 61.9%
    8k  truncated= 81  answer-in-trace= 17  rescued= 11  captured= 64.7%
   16k  truncated= 58  answer-in-trace= 1

In [2]:
# ============================================================================
# PHASE 11b: CORRECTED CEILING - restricted to RESCUABLE (wrong) samples
# ============================================================================
import os, re, json
from transformers import AutoTokenizer

os.makedirs("outputs", exist_ok=True)
hits = []
for root in ("/kaggle/input", "/kaggle/working"):
    if os.path.exists(root):
        for dp, _, fns in os.walk(root, followlinks=True):
            for fn in fns:
                if fn.endswith(".json") and ("trace" in fn.lower() or "phase7" in fn.lower()):
                    hits.append(os.path.join(dp, fn))
hits = sorted(set(hits), key=lambda p: -os.path.getsize(p))
d = json.load(open(hits[0]))
truths, traces = d["truths"], d["traces"]
tok = AutoTokenizer.from_pretrained("WeiboAI/VibeThinker-3B")

K = 4
# rescued counts measured in phase 9 (8k, 16k) and phase 6 (4k)
RESCUED = {4096: 13, 8192: 11, 16384: 9}

def all_boxed(t):
    out, i = [], 0
    while True:
        i = t.find("\\boxed{", i)
        if i == -1: return out
        j, dep, buf = i + 7, 1, []
        while j < len(t) and dep > 0:
            ch = t[j]
            if ch == "{": dep += 1
            elif ch == "}":
                dep -= 1
                if dep == 0: break
            buf.append(ch); j += 1
        out.append("".join(buf).strip()); i = j

def as_int(s):
    if s is None: return None
    m = re.search(r"-?\d+", s.replace(",", ""))
    return int(m.group()) if m else None

def prefix(t, B):
    if len(t["ids"]) <= B: return t["text"], (t["finish"] != "stop")
    return tok.decode(t["ids"][:B]), True

print(f"{'budget':>7}{'trunc':>8}{'already_ok':>12}{'RESCUABLE':>11}"
      f"{'has_ans':>9}{'FP':>6}{'net_ceil':>10}{'rescued':>9}{'captured':>10}")
print("-" * 84)
rows = []
for B in sorted(RESCUED):
    n_tr = n_ok = n_wrong = has = fp = 0
    for pi in range(30):
        truth, decoy = truths[pi], truths[(pi + 7) % 30]
        for ci in range(K):
            txt, truncated = prefix(traces[pi][ci], B)
            if not truncated: continue
            n_tr += 1
            boxes = all_boxed(txt)
            if (as_int(boxes[-1]) if boxes else None) == truth:
                n_ok += 1              # already correct -> NOT rescuable
                continue
            n_wrong += 1
            tail = txt[int(len(txt) * 0.8):]
            has += bool(re.search(rf"(?<!\d){truth}(?!\d)", tail))
            fp  += bool(re.search(rf"(?<!\d){decoy}(?!\d)", tail))
    net = max(has - fp, 0)             # FP-corrected ceiling
    r = RESCUED[B]
    cap = 100 * r / net if net else float("nan")
    print(f"{B//1024:>6}k{n_tr:>8}{n_ok:>12}{n_wrong:>11}{has:>9}{fp:>6}"
          f"{net:>10}{r:>9}{cap:>9.1f}%")
    rows.append({"budget": B, "truncated": n_tr, "already_correct": n_ok,
                 "rescuable": n_wrong, "answer_in_tail": has, "false_pos": fp,
                 "net_ceiling": net, "rescued": r, "captured_pct": round(cap, 1)})
print("-" * 84)
print("RESCUABLE = truncated AND wrong. This is the pool forcing can act on.")
print("net_ceil  = has_ans - FP, i.e. wrong traces that still mention the answer")
print("captured  = rescued / net_ceil")
print()
print("captured > 70%  -> forcing is near the extraction ceiling")
print("captured < 40%  -> a better extractor has real headroom")
print()
print("NOTE: 4k mixes arms - forced 40.8% came from a genuine 4k run (baseline")
print("      27.5%), while this ceiling is trace-derived (baseline 30.0%).")
print("      Trust 8k and 16k; treat 4k as indicative only.")

json.dump({"corrected_ceiling": rows}, open("outputs/phase11b_ceiling.json","w"), indent=2)
print("\nsaved outputs/phase11b_ceiling.json")

 budget   trunc  already_ok  RESCUABLE  has_ans    FP  net_ceil  rescued  captured
------------------------------------------------------------------------------------
     4k     105          21         84       16     3        13       13    100.0%
     8k      81          17         64       16     2        14       11     78.6%
    16k      58          14         44       11     0        11        9     81.8%
------------------------------------------------------------------------------------
RESCUABLE = truncated AND wrong. This is the pool forcing can act on.
net_ceil  = has_ans - FP, i.e. wrong traces that still mention the answer
captured  = rescued / net_ceil

captured > 70%  -> forcing is near the extraction ceiling
captured < 40%  -> a better extractor has real headroom

NOTE: 4k mixes arms - forced 40.8% came from a genuine 4k run (baseline
      27.5%), while this ceiling is trace-derived (baseline 30.0%).
      Trust 8k and 16k; treat 4k as indicative only.

saved outputs